# **CardiAg: RESP Preprocessing**

Preliminary version of the pipeline for preprocessing raw RESP signal, which will be featured as part of the Brain-Body Analysis Special Interest Group (BBSIG). For the most recent version, please refer to https://martager.github.io/bbsig. 

The methods used in this pipeline were heavily inspired by the article: Engelen T, Schuhmann T, Sack AT, Tallon-Baudry C (2025) Cardiac, respiratory, and gastric rhythms independently modulate motor corticospinal excitability in humans. PLOS Biology 23(11): e3003478. https://doi.org/10.1371/journal.pbio.3003478

Jupyter notebook created by Marta Gerosa

Created on: 16 May 2025

Last update (by M. Gerosa): 22 April 2026

If you use any BBSIG pipeline in a publication, please cite us: *Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://martager.github.io/bbsig/*

In [23]:
############## Import modules ##############

import numpy as np
import pandas as pd
import os, json
import matplotlib.pyplot as plt 
from scipy.stats import zscore
from scipy.signal import find_peaks

# Import custom bbsig module (`CardiAg_resp_functions.py` should be in the same directory)
import CardiAg_resp_functions as resp_utils

## **Settings: optional pipeline steps**

This section defines a series of variables that can be set to `True` if the corresponding pipeline steps needs to be included:

* **RESP signal flipping** (sect. 2): checks whether a RESP signal is inverted, and if so, corrects for this inversion resulting into a steeper rising phase (inspiration) and more gradual descending phase (expiration). 
* **Manual correction of expiration/inspiration onsets** (sect. 5): activates UI for manual correction of extra, missed or falsely detected expiration/inspiration onsets, using the custom `RespEditorGUI`defined in `CardiAg_resp_functions.py`. This can also be used to annotate bad segments. Saves a JSON file with the corrected peaks and bad segments. 
* **Save respiratory phase vector** (sect. 7): exports a TSV file including the instantaneoys phase vectors.

In [24]:
############## Settings: optional pipeline steps ##############

# For S2: set whether (optional) pre-processing step of flipping RESP signal is needed (when inverted)
# By convention, expiration peaks and inspiration troughs
resp_flip = True            # for automatic detection of inversion
resp_flip_force = True      # forces RESP signal flipping

# For S5: set whether manual correction of RESP peaks/troughs is needed using custom RespEditorGUI
resp_manual_correct = True

# For S7: set whether data export of TSV file with instantaneous phase vector values are needed
# If manual correction has been performed, both phase vectors for uncorrected and manually corrected exp/insp onsets are saved
phase_vector_output = True

## **1. Data import and conversion**

This section imports the physiological data and metadata from the `_physio.tsv.gz` and sidecar `_physio.json` file from their BIDS directory and extracts the RESP data into a DataFrame (`resp_df`) and numpy array (`resp_arr`). In detail:
- Define the name and directory path for the BIDS-compatible `_physio.tsv.gz` and `_physio.json` files containing physiological data and metadata, respectively.
- Import, decompress and store the TSV file as a DataFrame (`physio_df`).
- Extract metadata about column names and sampling frequency from the JSON file.
- Rename columns for respiratory data to 'resp'.
- Extract the RESP data into a DataFrame (`resp_df`) and numpy array (`resp_arr`).
- Define the path for storing RESP pre-processed data (`/derivatives/resp-preproc/`).  

In [ ]:
################## 1. Data import and conversion ##################

############## Define path for RESP data ##############

# Define the participant ID
participant_ids = [1]  # Adjust as needed: it should correspond to <ID> of 'sub-<ID>' in BIDS format

# Specify the directory of data storage (containing BIDS-compatible raw data)
wd = r'.\data'

# Mandatory: BIDS entities (task, datatype)
task_name = 'CardiAgIBTask' 
datatype_name = 'beh'     # current datatype folder according to BIDS

# Optional: BIDS entities (session)
session_idx = None           # <label> of 'ses-<label>' in BIDS format, if available; otherwise, set to None


############## Load raw RESP data ##############

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj) # participant ID (in BIDS format)

    ############## Create base BIDS-compatible filename and directory ##############

    # If you have additional BIDS entities (e.g., 'run' or 'recording') you can change the base BIDS file name below
    # e.g., f'{subj_id}_ses-{session_idx}_task-{task_name}_run-{run_idx}_recording-{rec_name}' 
    if session_idx is not None: 
        bids_base_fname = f'{subj_id}_ses-{session_idx}_task-{task_name}' 
        bids_base_dir = os.path.join(subj_id, f'ses-{session_idx}', datatype_name) # base BIDS folders incl. session 'sub-<ID>/ses-<label>/<datatype>/'
    else:
        bids_base_fname = f'{subj_id}_task-{task_name}'
        bids_base_dir = os.path.join(subj_id, datatype_name) # base BIDS folders without session 'sub-<ID>/<datatype>/'
    

    # Specify the directory and name of the BIDS-compatible physio data TSV.GZ file
    tsv_fname = f'{bids_base_fname}_physio.tsv.gz'
    tsv_dir = os.path.join(wd, bids_base_dir, tsv_fname)

    # Specify the name and directory of the BIDS-compatible physio metadata JSON file
    json_fname = f'{bids_base_fname}_physio.json'
    json_dir = os.path.join(wd, bids_base_dir, json_fname)

    # Check if the files exist for the participant
    if not os.path.exists(tsv_dir) or not os.path.exists(json_dir):
        print(f"Data files for {subj_id} not found. Skipping.")
        continue

    # Extract physio metadata from JSON file
    with open(json_dir, 'r') as fjson:
        physio_metadata = json.load(fjson)      # load metadata from physio.json
    physio_col = physio_metadata['Columns']     # column names from tsv.gz file

    # Open physio data file and save as dataframe
    physio_df = pd.read_csv(tsv_dir, compression='gzip', header=None, index_col=False,
                            sep='\t', names=physio_col)     # decompress and read TSV file
    physio_df.rename(columns={'respiratory': 'resp'}, inplace=True) # rename columns

    sfreq = physio_metadata['SamplingFrequency'] # save sampling rate
    
    # Extract array and dataframe for RESP data
    if 'resp' in physio_df.columns:
       
        resp_df = physio_df['resp'].copy() # save RESP df (one data sample per row)
        resp_arr = resp_df.values # save array with RESP data samples
    

    ############## Define path for RESP preprocessed data ##############
    
    # Check if the BIDS directory 'derivatives/resp-preproc/sub-<ID>/...' exists, if not create it
    resp_preproc_folder = 'resp-preproc'
    resp_preproc_dir = os.path.join(wd, 'derivatives', resp_preproc_folder, bids_base_dir)
    if not os.path.exists(resp_preproc_dir):
        os.makedirs(resp_preproc_dir)

## **2. RESP flipping**

Check if the signal is inverted by checking the slope asymmetry of the ascending or descending phases (or force it, if `resp_flip_force` is set to `True`). Ideally, inspiration should correspond to the steeper ascending slope, culminating in the expiration onset as peak, while expiration should correspond to the descending slope, ending with the inspiration onset as trough. 

In [ ]:
################## 2. RESP inversion check ##################

# Proceed only if (optional) pre-processing step of RESP flipping is required
if resp_flip: 

    # Check if signal is inverted by checking the slope asimmetry of the ascending and descending phases
    # Ideally, inspiration should correspond to the steeper ascending slope, while expiration should correspond to the descending slope
    slope = np.gradient(resp_arr)
    ascending_slope = np.mean(slope[slope > 0])     # ascending slope
    descending_slope = np.mean(slope[slope < 0])    # descending slope

    # If the descending slope is greater than the ascending one, the signal is likely inverted
    is_inverted = abs(descending_slope) > abs(ascending_slope)

    # If True, invert the signal to have inspiration onset as troughs, expiration onset as peaks
    if is_inverted or resp_flip_force:
        resp_arr = -resp_arr

## **3. RESP cleaning**

Based on the method described in Engelen et al. (2024), clean the raw RESP signal according to the following steps: 
- **3a. Smoothing with window moving average**: smooth RESP signal using a 600 ms window moving average (equivalent to `movmean` in MATLAB)
- **3b. Z-scoring**: normalize the smoothed RESP signal by z-scoring

In [ ]:
################## 3. RESP signal cleaning (based on Engelen et al., 2024) ##################

############## 3a. Smoothing with window moving average ##############

# Choose moving window size (in sec and samples)
window_size_s = 0.6
window_size = int(window_size_s * sfreq)

# Smoothing with window moving average (e.g., 600 ms window, depending on worst participant)
resp_smooth = pd.Series(resp_arr).rolling(window=window_size, center=True, min_periods=1).mean().to_numpy()

############## 3b. Z-scoring ##############

# Z-scoring the RESP signal
resp_smooth_zscore = zscore(resp_smooth, nan_policy='omit')

## **4. Expiration and inspiration onsets detection**

Based on the method described in Engelen et al. (2024), find the expiration onsets (peaks) and inspiration onsets (troughs) according to the following steps: 
- **4a. Detect expiration onsets (peaks)**: find expiration onsets as peaks with minimum distance of 1 sec and minimum peak prominence equal to the mean peak positive height, excluding outliers over 3 SD from the mean. 
- **4b. Detect inspiration onsets (troughs)**: find inspiration onsets (troughs) using an adapted trapezoid area algorithm (Vazquez-Seisdedos et al., 2011) that searches backwards from each expiration onset for prior local max 1st derivative, then iteratively computing trapezoid areas and selects moving point that maximizes the area of the trapezoid. 
- **4c. Build respiratory cycle structure**: define each respiratory cycle, spanning from one expiration peak to the next one, including one inspiration trough. This is used to calculate the average respiratory cycle duration. 

In [ ]:
############## 4a. Detect expiration onsets (peaks) ##############

# Calculate the min peak prominence as the mean peak positive height, excl. outliers (> 3SD)
resp_mean = np.mean(resp_smooth_zscore)
resp_sd = np.std(resp_smooth_zscore)
valid_peaks = resp_smooth_zscore[(resp_smooth_zscore > resp_mean) 
                          & (resp_smooth_zscore <= (resp_mean + (3 * resp_sd)))]   # exclude outliers > 3SD to compute min peak prominence
min_peak_prom = np.mean(valid_peaks)

# Find expiration onsets (peaks) (min distance = 1 sec, mean peak prominence = see above)
min_peak_distance = 1000    # 1 sec in samples
exp_onsets, _ = find_peaks(resp_smooth_zscore, prominence=min_peak_prom, distance=min_peak_distance)

In [ ]:
############## 4b. Detect inspiration onsets (troughs) ##############

# Define function to find inspiration onsets (troughs) using an adapted trapezoid area algorithm (Vazquez-Seisdedos et al., 2011)
# searching backwards from each expiration onset for prior local max 1st derivative, then iteratively computing trapezoid areas
# and selecting moving point that maximizes the area of the trapezoid
def find_insp_onsets(exp_onsets, resp_1stder_smooth, resp_smooth_zscore):

    insp_onsets = np.full(len(exp_onsets), np.nan)
    
    for i_exps in range(1, len(exp_onsets)):
             
        # Iteratively adjust window size (from 70% to 50% of cycle length) to avoid finding local maximum at window start
        window_end = exp_onsets[i_exps]
        window_sizes = [0.7, 0.6, 0.5]

        for size in window_sizes:
            cycle_length = exp_onsets[i_exps] - exp_onsets[i_exps - 1]
            window_begin = int(round(exp_onsets[i_exps] - size * cycle_length))
            window_begin = max(window_begin, 0)

            current_win = resp_1stder_smooth[window_begin:window_end]
            if len(current_win) == 0:
                continue
            
            # Find Xm point as local maximum in 1st derivative backwards from expiration onset 
            xm_relative = np.argmax(current_win)
            xm = xm_relative + window_begin

            if xm != window_begin:
                break  # found a valid Xm point different from window start; if not, repeat search by shrinking window size
        else:
            print(f"Cycle {i_exps}: Max derivative at window start even after shrinking.")
            continue  # Skip this cycle
       
        # Get reference points for trapezoid (Ym, Xr)
        ym = resp_smooth_zscore[xm] # y-value of Xm point
        xr = window_begin           # x-value at which search window begins
        window_length = xm - xr     # window length to determine nr of samples the trapeziums are fitted on
        
        # Initialize variables for trapezoid search (mobile Xi point)
        xi = []
        yi = []
        trap_area = []
        
        # Iterate backwards from Xm to window start in order to calculate trapezium areas
        for isearch in range(1, window_length):
            xi_val = xm - isearch
            
            # Ensure not to go below signal start
            if xi_val < 0:
                break
            
            # Ensure that the signal is not rising above Xm
            yi_val = resp_smooth_zscore[xi_val]
            if yi_val > ym:
                break
                
            # Calculate trapezoid area
            area = abs(0.5 * (ym - yi_val) * (2 * xr - xi_val - xm))
            trap_area.append(area)
            xi.append(xi_val)
            yi.append(yi_val)
        
        # Get Xi point that maximizes the trapezium area
        if trap_area:
            max_area_id = np.argmax(trap_area)
            insp_onsets[i_exps] = float(xi[max_area_id])
        else:
            print(f'No trapezium area found. {area}')
    
    # Get valid inspiration onset indices (delete None for first inspiration peak)
    # note that this method is always discarding the first cycle, 
    # because an exp peak is needed as a starting point to find backwards the insp trough
    insp_onsets = insp_onsets[~np.isnan(insp_onsets)].astype(int)
            
    return insp_onsets

In [ ]:
# Calculate the first derivative and smooth it 
resp_1stder = np.diff(resp_smooth_zscore)
resp_1stder_smooth = pd.Series(resp_1stder).rolling(
    window=window_size, center=True, min_periods=1).mean().to_numpy()

# Find inspiration onsets (troughs) using an adapted trapezoid area algorithm (Vazquez-Seisdedos et al., 2011)
insp_onsets = find_insp_onsets(exp_onsets=exp_onsets,
                               resp_1stder_smooth=resp_1stder_smooth,
                               resp_smooth_zscore=resp_smooth_zscore)

In [ ]:
############## 4c. Build respiratory cycle structure ############## 

# Store uncorrected expiration and inspiration onsets in a dict
resp_preproc_data = {
    'exp_onsets': {'Uncorr': exp_onsets}, 
    'insp_onsets': {'Uncorr': insp_onsets}, 
    'resp_cycles': {}}

# Build respiratory cycle structure (from one expiration peak to the next one, encompassing one inspiration trough)
resp_cycles = []

for irespcycles in range(len(exp_onsets) - 1):
    cur_exp_onset = exp_onsets[irespcycles]
    next_exp_onset = exp_onsets[irespcycles + 1]
    
    # Find expiration sample between current and next inspiration
    cur_insp_onset = insp_onsets[(insp_onsets > cur_exp_onset) & (insp_onsets < next_exp_onset)]
    if len(cur_insp_onset) == 1:
        if not (np.isnan(cur_exp_onset) or np.isnan(next_exp_onset) or np.isnan(cur_insp_onset)):
            resp_cycles.append({
                'resp_onset': cur_exp_onset,
                'insp_onset': cur_insp_onset[0],
                'resp_end': next_exp_onset - 1
            })

resp_preproc_data['resp_cycles']['Uncorr'] = resp_cycles

# Calculate respiratory cycle durations in seconds
exp_lengths = np.array([cycle['insp_onset'] - cycle['resp_onset'] for cycle in resp_cycles]) / sfreq
insp_lengths = np.array([cycle['resp_end'] - cycle['insp_onset'] for cycle in resp_cycles]) / sfreq

In [ ]:
# Plot histogram of uncorrected respiratory phase lengths for each subject
plt.figure(figsize=(5,5), dpi=300)
plt.hist(exp_lengths, alpha=0.7, label='Expiration', bins=30, edgecolor='black')
plt.hist(insp_lengths, alpha=0.7, label='Inspiration', bins=30, edgecolor='black')
plt.tick_params(direction='out', labelsize='medium')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.legend(frameon=False)
plt.xlabel('Respiration cycle length (in sec)', fontsize='large')
plt.ylabel('Bin size', fontsize='large')
plt.title('Uncorrected Respiratory Cycle Duration per Phase', fontsize='large')
plt.show()

## **5. (Optional) expiration/inspiration onset manual correction**

If enabled via `resp_manual_correct`, this section allows the interactive manual correction of expiration & inspiration onset locations and identification of noisy segments in the RESP signal using a custom-made `RespEditorGUI`, defined in the associated module `CardiAg_resp_functions.py`. The GUI plots the smoothed, z-scored RESP signal with expiration onsets (blue dots) and inspiration onsets (red dots). It allows deleting extra or falsely detected expiration/inspiration onsets through right-clicking the respective dots, and adding new expiration/inspiration onsets locations by selecting the desired area where the local maxima or minima will be selected, respectively (using the `Add Expiration Peaks` or `Add Inspiration Troughs` buttons). This interactive plot also allows the possibility to annotate noisy segments, using the `Annotate Bad Segment` button. After saving the data (using the `Save All` button), the corrected expiration/inspiration onset locations, as well as bad segments index pairs, are saved to a JSON file (`resp-preproc_manualcorr.json`) for further processing and analysis. In detail: 

- **5a. (Optional) expiration/inspiration onset manual correction**: if `resp_manual_correct` is set to `True`, enables the manual correction of expiration and inspiration onsets using the custom-made `RespEditorGUI`. 
- **5b. Load manually corrected data and quality summary**: loads the manually corrected RESP data to calculate a summary of how many expiration and inspiration onset locations were manually corrected (and where in the recording), and how many were unchanged from the automated algorithm. 

In [ ]:
############## 5a. (Optional) expiration/inspiration onset manual correction ##############

# Proceed only if (optional) preprocessing step of manual correction is needed
if resp_manual_correct: 

    # Display interactive plot
    %matplotlib ipympl

    # Display GUI for manual correction of expiration/inspiration onsets and annotation of bad segments
    resp_utils.RespEditorGUI(signal=resp_smooth_zscore,                             # smoothed z-scored RESP signal
                            exp_onsets=resp_preproc_data['exp_onsets']['Uncorr'],   # array of expiration onsets (peaks) 
                            insp_onsets=resp_preproc_data['insp_onsets']['Uncorr'], # array of inspiration onsets (troughs)
                            sfreq=sfreq,                                            # sampling frequency
                            save_dir=resp_preproc_dir,                              # directory of storage of preprocessed RESP data
                            bids_base_fname=bids_base_fname,                        # BIDS-compatible base filename (see S1)
                            figsize=(16,8))                                         # figure size in inches

In [ ]:
############## 5b. Load manually corrected data and quality summary ##############

if resp_manual_correct:
    
    resp_mancorr_fname = f'{bids_base_fname}_resp-preproc_manualcorr.json'
    resp_mancorr_dir = os.path.join(resp_preproc_dir, resp_mancorr_fname)

    # Check if JSON file exists and, if so, load it
    if os.path.exists(resp_mancorr_dir):
        with open(resp_mancorr_dir, 'r') as resp_mancorr:
            resp_mancorr_data = json.load(resp_mancorr)
            
    # Extract indices of manually corrected expiration and inspiration onsets
    exp_onsets_mancorr = np.array(resp_mancorr_data['expiration_onsets'])
    insp_onsets_mancorr = np.array(resp_mancorr_data['inspiration_onsets'])
    
    # Store manually corrected expiration/inspiration onsets
    resp_preproc_data['exp_onsets']['ManualCorr'] = exp_onsets_mancorr
    resp_preproc_data['insp_onsets']['ManualCorr'] = insp_onsets_mancorr

    # Build respiratory cycle structure (from one expiration peak to the next one, encompassing one inspiration trough)
    resp_cycles_mancorr = []

    for irespcycles in range(len(exp_onsets_mancorr) - 1):
        cur_exp_onset = exp_onsets_mancorr[irespcycles]
        next_exp_onset = exp_onsets_mancorr[irespcycles + 1]
        
        # Find expiration sample between current and next inspiration
        cur_insp_onset = insp_onsets_mancorr[(insp_onsets_mancorr > cur_exp_onset) & (insp_onsets_mancorr < next_exp_onset)]
        if len(cur_insp_onset) == 1:
            if not (np.isnan(cur_exp_onset) or np.isnan(next_exp_onset) or np.isnan(cur_insp_onset)):
                resp_cycles_mancorr.append({
                    'resp_onset': cur_exp_onset,
                    'insp_onset': cur_insp_onset[0],
                    'resp_end': next_exp_onset - 1
                })

    resp_preproc_data['resp_cycles']['ManualCorr'] = resp_cycles_mancorr

    # Calculate respiratory cycle durations in seconds
    exp_lengths_mancorr = np.array([cycle['insp_onset'] - cycle['resp_onset'] for cycle in resp_cycles_mancorr]) / sfreq
    insp_lengths_mancorr = np.array([cycle['resp_end'] - cycle['insp_onset'] for cycle in resp_cycles_mancorr]) / sfreq


    ##### Data quality summary for manual correction of exp/insp onsets #####

    # Create sets for uncorrected and manually corrected exp/insp onsets
    exp_uncorr_set = set(resp_preproc_data['exp_onsets']['Uncorr'] )
    exp_manual_set = set(resp_preproc_data['exp_onsets']['ManualCorr'] )
    insp_uncorr_set = set(resp_preproc_data['insp_onsets']['Uncorr'] )
    insp_manual_set = set(resp_preproc_data['insp_onsets']['ManualCorr'] )

    # Find total N of manually corrected (i.e., added or removed) exp peaks and insp troughs
    mancorr_peaks = len(exp_manual_set - exp_uncorr_set) + len(exp_uncorr_set - exp_manual_set)
    mancorr_troughs = len(insp_manual_set - insp_uncorr_set) + len(insp_uncorr_set - insp_manual_set)
    matched = len(exp_uncorr_set & exp_manual_set) + len(insp_uncorr_set & insp_manual_set)

    print("---------- Data quality summary: manual correction ----------")
    print(f"Manually corrected expiration peaks: {mancorr_peaks} indices")
    print(f"Manually corrected inspiration troughs: {mancorr_troughs} indices")
    print(f"Unchanged peaks/troughs after manual correction: {matched} indices")

In [ ]:
def samples_to_minutes(samples):
    return [round(x / sfreq, 3) for x in samples]  # rounded to 3 decimals

print("Peaks added (min):", samples_to_minutes(sorted(exp_manual_set - exp_uncorr_set)))
print("Peaks removed (min):", samples_to_minutes(sorted(exp_uncorr_set - exp_manual_set)))
print("Troughs added (min):", samples_to_minutes(sorted(insp_manual_set - insp_uncorr_set)))
print("Troughs removed (min):", samples_to_minutes(sorted(insp_uncorr_set - insp_manual_set)))

In [ ]:
if resp_manual_correct: 
    
    # Close interactive backend
    plt.close('all')
    %matplotlib inline

    # Plot histogram of uncorrected respiratory phase lengths
    plt.figure(figsize=(5,5), dpi=300)
    plt.hist(exp_lengths_mancorr, alpha=0.7, label='Expiration', bins=30, edgecolor='black')
    plt.hist(insp_lengths_mancorr, alpha=0.7, label='Inspiration', bins=30, edgecolor='black')
    plt.tick_params(direction='out', labelsize='medium')
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.legend(frameon=False)
    plt.xlabel('Respiration cycle length (in sec)', fontsize='large')
    plt.ylabel('Bin size', fontsize='large')
    plt.title('Manually Corrected Respiratory Cycle Duration per Phase', fontsize='large')
    plt.show()

## **6. Respiratory phase vector computation**

Based on the method described in Engelen et al. (2024), assign a respiratory phase vector to the respiratory cycles: segments between exp onset and insp onset will be assigned 0-pi, while segments in between insp onset and next exp onset will be assigned pi-2pi. 

In [ ]:
############## 6. Respiratory phase vector computation ############## 

# Assign a phase vector: segments between exp onset and insp onset will be assigned 0-pi, 
# segments in between insp onset and next exp onset will be assigned pi-2pi
resp_phase_vec = np.full(resp_smooth_zscore.shape, np.nan)
resp_insp_starts = np.array([cycle['insp_onset'] for cycle in resp_cycles])
resp_cycle_starts = np.array([cycle['resp_onset'] for cycle in resp_cycles])
resp_cycle_ends = np.array([cycle['resp_end'] for cycle in resp_cycles])

for ncycles_resp in range(len(resp_cycles)):
    resp_on = int(resp_cycle_starts[ncycles_resp])
    insp_on = int(resp_insp_starts[ncycles_resp])
    resp_off = int(resp_cycle_ends[ncycles_resp])

    t1 = np.arange(resp_on, insp_on + 1)
    resp_phase_vec[resp_on:insp_on + 1] = np.pi * (t1 - resp_on) / (insp_on - resp_on)

    t2 = np.arange(insp_on + 1, resp_off + 1)
    resp_phase_vec[insp_on + 1:resp_off + 1] = np.pi + np.pi * (t2 - insp_on) / (resp_off - insp_on)

# Store respiratory phase vector in a dataframe
resp_phase_vec_df = pd.DataFrame(resp_phase_vec, columns=['phase_vector_uncorr'])

In [ ]:
# If manual correction was performed, compute the respective phase vector for manually corrected exp/insp onsets
if resp_manual_correct:

    # Assign a phase vector: segments between exp onset and insp onset will be assigned 0-pi, 
    # segments in between insp onset and next exp onset will be assigned pi-2pi
    resp_phase_vec_mancorr = np.full(resp_smooth_zscore.shape, np.nan)
    resp_insp_starts_mancorr = np.array([cycle['insp_onset'] for cycle in resp_cycles_mancorr])
    resp_cycle_starts_mancorr = np.array([cycle['resp_onset'] for cycle in resp_cycles_mancorr])
    resp_cycle_ends_mancorr = np.array([cycle['resp_end'] for cycle in resp_cycles_mancorr])

    for ncycles_resp in range(len(resp_cycles_mancorr)):
        resp_on_mancorr = int(resp_cycle_starts_mancorr[ncycles_resp])
        insp_on_mancorr = int(resp_insp_starts_mancorr[ncycles_resp])
        resp_off_mancorr = int(resp_cycle_ends_mancorr[ncycles_resp])

        t1_mancorr = np.arange(resp_on_mancorr, insp_on_mancorr + 1)
        resp_phase_vec_mancorr[resp_on_mancorr:insp_on_mancorr + 1] = np.pi * (t1_mancorr - resp_on_mancorr) / (insp_on_mancorr - resp_on_mancorr)

        t2_mancorr = np.arange(insp_on_mancorr + 1, resp_off_mancorr + 1)
        resp_phase_vec_mancorr[insp_on_mancorr + 1:resp_off_mancorr + 1] = np.pi + np.pi * (t2_mancorr - insp_on_mancorr) / (resp_off_mancorr - insp_on_mancorr)

    # Store respiratory phase vector for manually corrected data in a dataframe
    resp_phase_vec_df['phase_vector_manualcorr'] = resp_phase_vec_mancorr

## **7. Data output**

This section exports the RESP preprocessing output files in BIDS-compatible format for each subject in `derivatives/resp-preproc/sub-xx/`.

- **7a. (Optional) Create and export TSV.GZ file with raw and clean RESP data**: a custom function `save_resp_cleaned()` saves the BIDS-compatible `_resp-cleaned.tsv.gz` file having two columns, `resp_raw` (i.e., the original RESP data)and `resp_cleaned` (i.e., the smoothed and z-scored RESP signal, created in S3).
- **7b. Export main RESP preprocessing info to JSON file**: a custom function `save_resp_preproc()` saves the `_resp-preproc.json` file with expiration onset indices, inspiration onset indices, respiratory cycle structures, bad segments indices and other relevant metadata. 
    - `exp_onsets` contains the list of indices for `Uncorr` and (if present) `ManualCorr` expiration onsets (peaks)
    - `insp_onsets` contains the list of indices for `Uncorr` and (if present) `ManualCorr` inspiration onsets (troughs)
    - `resp_cycles` contains a list of dictionaries including indices of individual respiratory cycles, including respiratory cycle onsets/offsets 
    and intermediate inspiration onsets, for `Uncorr` and (if present) `ManualCorr` data
    - `bad_segments` contains idx pairs indicating onsets & offsets of bad segments from manual correction (if present)
    - `info` contains metadata about sampling frequency, recording length, and expiration/inspiration onset detection (e.g., method, N of corrected peaks & troughs)
- **7c. (Optional) Export TSV.GZ file with instantaneous phase vector(s)**: a custom function `save_phase_vector()` saves the `_resp-phase-vector.tsv.gz` file with respiratory phase vector (from uncorrected and, optionally, manually corrected data) to a TSV.GZ file. 

In [ ]:
############## 7. Data output ##############

############# 7a. Create and export TSV.GZ file with raw and clean RESP data  ##############

# Define function to save TSV.GZ file containing raw and cleaned RESP data
def save_resp_cleaned(resp_raw, resp_clean):
    """ Save raw and cleaned RESP data to a TSV.GZ file.

    Parameters:
    - resp_raw (ndarray): Array containing raw RESP data (i.e., same as BIDS-compatible _physio.tsv.gz).
    - resp_clean (ndarray): Array containing cleaned RESP data after applying smoothing and z-scoring.
    """

    # Merge resp_raw and ecg_clean into one dataframe
    resp_all = pd.DataFrame({'resp_raw': resp_raw, 
                             'resp_cleaned': resp_clean})

    # Save as TSV.GZ in derivatives/ecg-preproc/sub-XX/beh/ folder 
    resp_all_fname = f'{bids_base_fname}_resp-cleaned.tsv.gz'
    resp_all_fpath = os.path.join(resp_preproc_dir, resp_all_fname)
    resp_all.to_csv(resp_all_fpath, index=False, compression='gzip',
                    sep='\t', na_rep="n/a")
    
    print(f"RESP cleaned TSV.GZ file has been saved to {resp_all_fpath}")

In [ ]:
############# Save TSV.GZ file with raw and clean RESP data  ##############

# Save TSV.GZ file including raw and clean RESP data (according to Engelen et al., 2024)
save_resp_cleaned(resp_raw=resp_arr, resp_clean=resp_smooth_zscore)

In [ ]:
############# 7b. Export main RESP preprocessing info to JSON file  ##############

# Function to convert values to int or None
def convert_to_int_or_none(value):
    if isinstance(value, (float, int, np.integer, np.floating)):
        return int(value) if not np.isnan(value) else None
    return None

# Create a function for data export of main RESP preprocessing info
def save_resp_preproc(resp_preproc_data, resp_clean, resp_manual_correct=resp_manual_correct):
    """ Save RESP preprocessing information to a JSON file. 

    Parameters:
    - resp_preproc_data (dict): Dictionary containing expiration onsets, inspiration onsets and respiratory cycles
        from uncorrected and, if previously enabled, manually corrected data.
    - resp_clean (array): Array containing cleaned RESP data after applying smoothing and z-scoring.
    - resp_manual_correct (bool): whether manual correction of exp peaks and insp troughs was performed with RespEditorGUI.

    Returns:
    - dict: dictionary containing all relevant RESP preprocessing information:
        - 'exp_onsets': list of idx for 'Uncorr' and (if present) 'ManualCorr' expiration onsets (peaks)
        - 'insp_onsets': list of idx for 'Uncorr' and (if present) 'ManualCorr' inspiration onsets (troughs)
        - 'resp_cycles': list of dictionaries including idx of individual respiratory cycles, incl. respiratory cycle onsets/offsets 
            and intermediate inspiration onsets, for 'Uncorr' and (if present) 'ManualCorr' data
        - 'bad_segments': list of idx pairs indicating onsets & offsets of bad segments from manual correction (if present)
        - 'info': metadata about sampling frequency, recording length, exp/insp onset detection (e.g., method, nr of corrected peaks & troughs)
    """

    # Intialize dict for saving main RESP preprocessing info
    resp_preproc_dict = {'exp_onsets': {'Uncorr': {}}, 
                        'insp_onsets': {'Uncorr': {}}, 
                        'resp_cycles': {'Uncorr': {}},
                        'bad_segments': {},
                        'info': {}}
   
    ###### Store exp/insp onsets & respiratory cycles from uncorrected detection ######

    # Extract indices of uncorrected expiration and inspiration onsets 
    exp_onsets_uncorr = [convert_to_int_or_none(x) for x in resp_preproc_data['exp_onsets']['Uncorr']]
    insp_onsets_uncorr = [convert_to_int_or_none(x) for x in resp_preproc_data['insp_onsets']['Uncorr']]

    # Store uncorrected expiration/inspiration onsets
    resp_preproc_dict['exp_onsets']['Uncorr'] = exp_onsets_uncorr
    resp_preproc_dict['insp_onsets']['Uncorr'] = insp_onsets_uncorr

    # Store information about onset/offset of each uncorrected respiratory cycle, plus intermediate insp onset
    resp_preproc_dict['resp_cycles']['Uncorr'] = []  
    for cycle in resp_cycles:
        resp_preproc_dict['resp_cycles']['Uncorr'].append({
            'resp_onset': convert_to_int_or_none(cycle['resp_onset']),
            'insp_onset': convert_to_int_or_none(cycle['insp_onset']),
            'resp_end': convert_to_int_or_none(cycle['resp_end'])
        })


    ##### Load manually corrected expiration/inspiration onsets #####
    
    if resp_manual_correct:
        
        # Load file with information about manual correction
        resp_mancorr_fname = f'{bids_base_fname}_resp-preproc_manualcorr.json'
        resp_mancorr_dir = os.path.join(resp_preproc_dir, resp_mancorr_fname)

        # Check if JSON file exists and, if so, load it
        if os.path.exists(resp_mancorr_dir):
            with open(resp_mancorr_dir, 'r') as resp_mancorr:
                resp_mancorr_data = json.load(resp_mancorr)
                
        # Extract indices of manually corrected expiration and inspiration onsets
        exp_onsets_mancorr = [convert_to_int_or_none(x) for x in resp_preproc_data['exp_onsets']['ManualCorr']] 
        insp_onsets_mancorr = [convert_to_int_or_none(x) for x in resp_preproc_data['insp_onsets']['ManualCorr']]
        
        # Store manually corrected expiration/inspiration onsets
        resp_preproc_dict['exp_onsets']['ManualCorr'] = exp_onsets_mancorr
        resp_preproc_dict['insp_onsets']['ManualCorr'] = insp_onsets_mancorr

        # Store information about onset/offset of each uncorrected respiratory cycle, plus intermediate insp onset
        resp_preproc_dict['resp_cycles']['ManualCorr'] = []  
        for cycle in resp_preproc_data['resp_cycles']['ManualCorr']:
            resp_preproc_dict['resp_cycles']['ManualCorr'].append({
                'resp_onset': convert_to_int_or_none(cycle['resp_onset']),
                'insp_onset': convert_to_int_or_none(cycle['insp_onset']),
                'resp_end': convert_to_int_or_none(cycle['resp_end'])
            })

        # Store bad segments
        bad_segments_idx = resp_mancorr_data.get('bad_segments', [])
        resp_preproc_dict['bad_segments'] = bad_segments_idx


    # Store metadata (sampling frequency, recording length and method)
    resp_preproc_dict['info']['sfreq'] = sfreq
    resp_preproc_dict['info']['recording_length'] = len(resp_clean)
    resp_preproc_dict['info']['method'] = 'Engelen et al. (2024). doi: https://doi.org/10.1101/2024.09.10.612221'

    # Store metadata about manually corrected peaks/troughs
    if resp_manual_correct:
        resp_preproc_dict['info']['manualcorr_n_peaks'] = mancorr_peaks
        resp_preproc_dict['info']['manualcorr_n_troughs'] = mancorr_troughs


    ###### Save JSON file with main info from RESP preprocessing ######

    resp_preproc_json_fname = f'{bids_base_fname}_resp-preproc.json'
    resp_preproc_json_fpath = os.path.join(resp_preproc_dir, resp_preproc_json_fname)
    
    with open(resp_preproc_json_fpath, 'w') as resp_preproc_json:
        json.dump(resp_preproc_dict, resp_preproc_json, allow_nan=True, indent=4)
    
    print(f"RESP preprocessing results have been saved to {resp_preproc_json_fpath}")

    return resp_preproc_dict

In [ ]:
############# Export RESP preprocessing JSON with selected options ##############
resp_preproc_dict = save_resp_preproc(resp_preproc_data=resp_preproc_data, resp_clean=resp_smooth_zscore)

In [ ]:
############# 7c. (Optional) Export TSV.GZ file with instantaneous phase vector(s) ##############

def save_phase_vector(resp_phase_vec_df):
    """ Save respiratory phase vector (from uncorrected and, optionally, manually corrected data) to a TSV.GZ file.

    Parameters:
    - resp_phase_vec_df (DataFrame): DataFrame containing respiratory phase vector computed from uncorrected 
        and, optionally, manually corrected exp/insp onsets, where expiration is assigned values from 0 to pi, 
        while inspiration from pi to 2pi. The DataFrame has same length as the original raw RESP recording. 
    """
    
    # Save as TSV.GZ in derivatives/ecg-preproc/sub-XX/beh/ folder 
    phasevec_fname = f'{bids_base_fname}_resp-phase-vector.tsv.gz'
    phasevec_fpath = os.path.join(resp_preproc_dir, phasevec_fname)
    resp_phase_vec_df.to_csv(phasevec_fpath, index=False, compression='gzip',
                    sep='\t', na_rep="n/a")

    print(f"Respiratory phase vector TSV.GZ file has been saved to {phasevec_fpath}")

In [ ]:
# If the optional manual correction step has been performed, the TSV.GZ file will include two
# columns 'phase_vector_uncorr' and 'phase_vector_manualcorr'; otherwise, only the former will be included
if phase_vector_output: 
    hr_bpm_df = save_phase_vector(resp_phase_vec_df=resp_phase_vec_df)

## **Final note**

The current script was a preliminary development version of the RESP Preprocessing pipeline from the Brain-Body Analysis Special Interest Group (BBSIG), which will be launched later in 2026. For the most recent version, we recommend to check out our repository: https://martager.github.io/bbsig/code/preprocessing/resp_preproc.ipynb

When using or adapting this BBSIG pipeline to conduct RESP preprocessing in your research work, please cite us in your publication as follows: 

**APA**

*Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://doi.org/10.5281/zenodo.15212797*